# BIOSTAT 707 Checkpoint 1

**Author:** Mai Li  
**NetID:** ml783

This notebook characterizes the PhysioNet Challenge 2012 set-a ICU cohort. Running it regenerates every CSV, figure, and this HTML report. Each of the 4,000 source files represents one ICU admission, and observations cover the first 48 hours. The accompanying Python script performs the processing; this notebook is the single execution entry point. The original files are read-only.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
assert (ROOT / 'checkpoint1.ipynb').exists()
subprocess.run(['python', 'scripts/run_analysis.py'], check=True)


## Cohort description and outcomes

Table 1 reports one observation per admission: baseline age, gender, ICU type, height, and the first recorded weight. Continuous distributions are reported as median [Q1, Q3] because several are asymmetric; categories use n (%) among nonmissing values. Every row gives valid and missing admission counts. Repeated time-series measurements are never counted as independent patients. Outcomes are joined one-to-one by RecordID. In the source, `-1` denotes unavailable information; it remains visible in the long table's `value_raw` column but is missing in numeric analyses. In particular, Survival = -1 is unavailable follow-up, not a negative survival time.


In [ ]:
display(pd.read_csv(ROOT / 'output/table1.csv'))
display(pd.read_csv(ROOT / 'output/outcome_summary.csv'))
display(pd.read_csv(ROOT / 'output/death_summary.csv'))


## Data quality and derived tables

The long table retains the original HH:MM time, numeric hours, variable, raw value, and cleaned numeric value. The [PhysioNet Challenge documentation](https://physionet.org/content/challenge-2012/1.0.0/) states that valid values are non-negative, `-1` means missing or unknown, arterial pH uses a 0–14 scale, GCS uses 3–15, FiO2 uses 0–1, and the categorical fields have specified codes. Those documented rules distinguish demonstrably invalid values from observations that are merely extreme.

We convert documented `-1` values, other negative measurements, and documented scale/code violations to missing in the numeric `value` column while retaining the original text in `value_raw`. We do not invent clinical cutoffs. Values such as Height from 1.8 to 431.8 cm, Urine up to 11,000 mL, and HR up to 300 bpm are flagged as suspicious but retained because the documentation warns that occasional outliers should be expected and does not give invalidity thresholds for them. Records sharing a RecordID, variable, and time are also retained: PhysioNet notes that different methods or sensors may produce measurements at the same or nearly the same time. The anomaly-decision table reports affected rows, actions, and reasons.

The wide table has one row per admission. Static fields are kept once; dynamic variables receive count, first, last, minimum, maximum, and mean over 0–48 hours. A zero count means no valid measurement, while other summaries remain missing. A single 48-hour window is simple and preserves broad coverage but loses timing; a later model may use separate 0–24 and 24–48 hour summaries.


In [ ]:
long = pd.read_csv(ROOT / 'output/set-a_long.csv')
wide = pd.read_csv(ROOT / 'output/set-a_wide.csv')
checks = {'long_rows': len(long), 'unique_records': long.RecordID.nunique(), 'variables': long.Parameter.nunique(), 'time_min': long.time_hours.min(), 'time_max': long.time_hours.max(), 'wide_rows': len(wide)}
display(pd.Series(checks, name='value').to_frame())
display(pd.read_csv(ROOT / 'output/data_quality_summary.csv'))
display(pd.read_csv(ROOT / 'output/anomaly_decisions.csv'))
display(pd.read_csv(ROOT / 'output/variable_quality.csv').query('documented_invalid_n > 0 or minus_one_n > 0 or duplicate_rows_n > 0').sort_values('duplicate_rows_n', ascending=False))


## Missingness maps

Calling `isna()` only on existing long-table rows would underestimate missingness because no row is written when a measurement was not taken. We therefore create the full grid of 4,000 admissions, 37 dynamic variables, and eight six-hour intervals. A cell is measured when at least one valid value occurs in that interval; otherwise it is unmeasured. Six hours balances temporal resolution and readability, and a measurement at exactly 48:00 is assigned to the final interval.

The summary table reports the percentage of admissions ever measured, mean number of recorded rows per admission, measurement percentage in hours 0–6 and 42–48, and their change. It makes clear that monitoring variables are measured for many patients and repeatedly, whereas tests such as troponin and cholesterol are ordered for small subsets. Changes across the first and last intervals show that missingness also varies over time. This likely reflects clinical workflow, test indication, monitoring method, and patient condition rather than random data loss. The maps visualize admission-level and cohort-level patterns; they do not imply that every variable should be measured every six hours.

The wide-table map shows the first 400 admissions and 60 features with the highest missing proportions. We leave missing values missing at this checkpoint rather than substituting zero. Later modeling may use missingness indicators and fit imputation using training data only. Differences in mortality between measured and unmeasured groups are descriptive evidence that missingness may be informative, not causal effects of testing.


In [ ]:
long_missing = pd.read_csv(ROOT / 'output/long_missingness_summary.csv')
display(long_missing.sort_values('admissions_measured_pct').head(10))
display(long_missing.sort_values('change_first_to_last_pct_points').head(10))
display(Image(filename=str(ROOT / 'output/figures/long_missingness_map.png'), alt='Measurement presence across admissions and six-hour intervals'))
display(Image(filename=str(ROOT / 'output/figures/variable_time_measurement_rate.png'), alt='Measurement rate by variable and six-hour interval'))
display(Image(filename=str(ROOT / 'output/figures/wide_missingness_map.png'), alt='Missing wide-table features by admission'))
display(pd.read_csv(ROOT / 'output/wide_missingness_summary.csv').head(20))
display(pd.read_csv(ROOT / 'output/missingness_by_outcome.csv').sort_values('never_measured_n', ascending=False).head(20))


## Key findings

In the long table, the documented missing code `-1` occurred 2,223 times, 24 other negative values occurred in `Temp`, and 9 pH values were outside the documented 0–14 range. The negative values and out-of-range pH values were converted to missing numeric values while their original text was retained. Repeated admission–variable–time keys affected 7,474 rows and were retained because the source allows measurements from different methods or sensors at the same time. The temporal analysis shows that monitoring variables are measured for most admissions, whereas tests such as Troponin I and Cholesterol are measured in only small subsets. Many laboratory measurements are less common in hours 42–48 than in hours 0–6, so missingness varies by variable and time and may reflect clinical measurement decisions rather than random data loss.

The wide table contains one row per admission and summarizes dynamic measurements over the full 0–48-hour period with count, first, last, minimum, maximum, and mean. This single-window choice is transparent and preserves cohort coverage, but it does not retain early-versus-late timing; separate 0–24 and 24–48-hour summaries are a reasonable later sensitivity analysis. Missing summaries remain missing, while count zero identifies admissions without a valid measurement. The wide-table missingness map shows that rare tests and their derived summaries are most incomplete. These patterns should be treated as potentially informative in later modeling, with missingness indicators or training-set-only imputation considered rather than replacing missing values with zero.


## Limitations and next steps

This is descriptive work on the first 48 hours, not a causal analysis or a prediction model. Some extreme values and repeated-time records remain in the summaries, potentially affecting minima, maxima, and means. The current wide table uses one 0–48-hour summary for each dynamic variable, which loses early-versus-late timing. Before modeling, we should verify units and value ranges against clinical documentation, compare 0–24-hour with 24–48-hour summaries, and perform sensitivity analyses for flagged values and informative missingness. The eventual model will be assessed using the smaller of sensitivity and positive predictive value, `min(TP/(TP+FN), TP/(TP+FP))`, and the Hosmer–Lemeshow statistic for calibration. No model or performance estimate is produced in this checkpoint.

## AI use statement

I used GPT/Codex to help me understand the assignment concepts, missing-data concepts, clinical variables, and the purpose of the requested analyses. I also used it to discuss analysis strategies and possible data-wrangling and visualization methods. I reviewed the resulting work and checked the project paths, input-file structure, RecordID matching, the interpretation of `-1` as missing, the handling of documented invalid values, the decision to retain flagged extreme values, the long-table and wide-table row counts, the outcome merge, and an admission-level HR summary against the long table. The final analysis choices and written conclusions were reviewed for consistency with the dataset documentation and the assignment requirements.
